In [1]:
import os
import time
import requests
import pandas as pd
from datetime import datetime
import psycopg2
from psycopg2.extras import execute_batch
from dotenv import load_dotenv

# =========================================================
# ENV
# =========================================================
load_dotenv()

API_KEY = os.getenv("FMP_API_KEY")

POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_HOST = os.getenv("POSTGRES_HOST", "localhost")
POSTGRES_PORT = os.getenv("POSTGRES_PORT", "5432")

if not API_KEY:
    raise EnvironmentError("❌ Falta FMP_API_KEY en .env")

if not all([POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD]):
    raise EnvironmentError("❌ Faltan variables de entorno de Postgres")

# =========================================================
# CONFIG
# =========================================================
SNAPSHOT_DATE = "2026-04-01"   # ← Mantener actualizado según tu ETL

# =========================================================
# RATE LIMIT CONFIG (300 req/min es el estándar de FMP)
# =========================================================
MAX_REQUESTS_PER_MIN = 300
REQUEST_WINDOW = 60
request_timestamps = []

# =========================================================
# CONEXIÓN DB
# =========================================================
conn = psycopg2.connect(
    dbname=POSTGRES_DB,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    host=POSTGRES_HOST,
    port=POSTGRES_PORT
)

# =========================================================
# OBTENER TICKERS
# =========================================================
def obtener_tickers(conn, snapshot_date):
    sql = """
        SELECT ticker
        FROM seleccion.universo_trabajo
        WHERE snapshot_date = %s
    """
    with conn.cursor() as cur:
        cur.execute(sql, (snapshot_date,))
        rows = cur.fetchall()
    return [r[0] for r in rows]

# =========================================================
# REQUEST SEGURO + RATE LIMIT
# =========================================================
def safe_get(url, retries=3, sleep_time=1.5):
    global request_timestamps
    for _ in range(retries):
        try:
            now = time.time()
            request_timestamps = [t for t in request_timestamps if now - t < REQUEST_WINDOW]

            if len(request_timestamps) >= MAX_REQUESTS_PER_MIN:
                sleep_for = REQUEST_WINDOW - (now - request_timestamps[0])
                if sleep_for > 0:
                    print(f"⏳ Rate limit alcanzado. Esperando {round(sleep_for,2)}s")
                    time.sleep(sleep_for)

            request_timestamps.append(time.time())
            r = requests.get(url, timeout=10)
            if r.status_code == 200:
                return r.json()
        except Exception as e:
            print(f"Error en request: {e}")
        time.sleep(sleep_time)
    return None

# =========================================================
# EXTRACT (Nuevo Endpoint Stable)
# =========================================================
def obtener_scores(ticker):
    """
    Usa el endpoint estable para obtener Altman Z-Score y Piotroski.
    """
    url = (
        f"https://financialmodelingprep.com/stable/financial-scores"
        f"?symbol={ticker}&apikey={API_KEY}"
    )
    return safe_get(url)

# =========================================================
# TRANSFORM
# =========================================================
def construir_row(ticker, response):
    if not response:
        return None

    # El endpoint /stable/ puede devolver una lista o un dict con clave 'data'
    data_list = response if isinstance(response, list) else response.get('data', [])
    
    if len(data_list) == 0:
        print(f"⚠️ No hay data de scores para {ticker}")
        return None

    # Tomamos el registro más reciente (index 0)
    data = data_list[0]
    now = datetime.utcnow()

    return {
        "ticker": ticker,
        "snapshot_date": SNAPSHOT_DATE,
        "altman_z_score": data.get("altmanZScore"),
        "piotroski_score": data.get("piotroskiScore"),
        "created_at": now,
        "updated_at": now
    }

# =========================================================
# INSERT DB (UPSERT)
# =========================================================
def insertar_scores(df, conn):
    # Asegúrate de que la tabla indicadores.scores exista
    sql = """
        INSERT INTO seleccion.scores (
            ticker,
            snapshot_date,
            altman_z_score,
            piotroski_score,
            created_at,
            updated_at
        )
        VALUES (%s,%s,%s,%s,%s,%s)
        ON CONFLICT (ticker, snapshot_date)
        DO UPDATE SET
            altman_z_score = EXCLUDED.altman_z_score,
            piotroski_score = EXCLUDED.piotroski_score,
            updated_at = EXCLUDED.updated_at
    """

    data = [
        (
            row["ticker"],
            row["snapshot_date"],
            row["altman_z_score"],
            row["piotroski_score"],
            row["created_at"],
            row["updated_at"]
        )
        for _, row in df.iterrows()
    ]

    with conn.cursor() as cur:
        execute_batch(cur, sql, data, page_size=500)
    conn.commit()

# =========================================================
# MAIN
# =========================================================
try:
    TICKERS = obtener_tickers(conn, SNAPSHOT_DATE)
    print(f"Total tickers a procesar: {len(TICKERS)}")

    rows = []
    for ticker in TICKERS:
        print(f"▶ Procesando {ticker}")
        raw = obtener_scores(ticker)
        row = construir_row(ticker, raw)
        if row:
            rows.append(row)

    if rows:
        df_final = pd.DataFrame(rows)
        print("\nResumen de carga (Top 5):")
        print(df_final[['ticker', 'altman_z_score', 'piotroski_score']].head())
        
        insertar_scores(df_final, conn)
        print(f"\n✅ {len(df_final)} registros guardados en indicadores.scores")
    else:
        print("\n⚠ No se generaron datos para insertar")

finally:
    conn.close()
    print("🔌 Conexión cerrada.")


Total tickers a procesar: 720
▶ Procesando A
▶ Procesando AA
▶ Procesando ABBV
▶ Procesando ABNB
▶ Procesando ABT
▶ Procesando ACA
▶ Procesando ACAD
▶ Procesando ACIC
▶ Procesando ACIW
▶ Procesando ACLS
▶ Procesando ACT
▶ Procesando ADBE
▶ Procesando ADI
▶ Procesando ADMA
▶ Procesando ADP
▶ Procesando ADUS
▶ Procesando AEIS
▶ Procesando AFG
▶ Procesando AGCO
▶ Procesando AGX
▶ Procesando AGYS
▶ Procesando AIG
▶ Procesando AII
▶ Procesando AIR
▶ Procesando AIT
▶ Procesando AIZ
▶ Procesando ALAB
▶ Procesando ALG
▶ Procesando ALGN
▶ Procesando ALL
▶ Procesando ALNT
▶ Procesando AMAT
▶ Procesando AMD
▶ Procesando AME
▶ Procesando AMKR
▶ Procesando AMSF
▶ Procesando AMZN
▶ Procesando ANIP
▶ Procesando AOS
▶ Procesando APA
▶ Procesando APEI
▶ Procesando APO
▶ Procesando APOG
▶ Procesando APPF
▶ Procesando APXT
▶ Procesando APXTU
▶ Procesando AR
▶ Procesando AREN
▶ Procesando ARLO
▶ Procesando ARLP
▶ Procesando ARW
▶ Procesando ARWR
▶ Procesando ASGN
▶ Procesando ASIC
▶ Procesando ASIX
▶ Proc

▶ Procesando ODC
▶ Procesando ODFL
▶ Procesando OFG
▶ Procesando OFLX
▶ Procesando OII
▶ Procesando OLED
▶ Procesando OLLI
▶ Procesando OPRX
▶ Procesando OPY
▶ Procesando ORI
▶ Procesando ORLY
▶ Procesando ORRF
▶ Procesando OSK
▶ Procesando OSPN
▶ Procesando OTTR
▶ Procesando OVBC
▶ Procesando OVV
▶ Procesando PANW
▶ Procesando PAY
▶ Procesando PAYC
▶ Procesando PAYO
▶ Procesando PAYS
▶ Procesando PB
▶ Procesando PBYI
▶ Procesando PCB
▶ Procesando PCTY
▶ Procesando PCYO
▶ Procesando PEBK
▶ Procesando PEGA
▶ Procesando PEN
▶ Procesando PFG
▶ Procesando PFIS
▶ Procesando PG
▶ Procesando PGNY
▶ Procesando PH
▶ Procesando PHIN
▶ Procesando PHM
▶ Procesando PINS
▶ Procesando PIPR
▶ Procesando PKE
▶ Procesando PKST
▶ Procesando PLAB
▶ Procesando PLMR
▶ Procesando PLNT
▶ Procesando PLOW
▶ Procesando PLPC
▶ Procesando PLTR
▶ Procesando PLXS
▶ Procesando PM
▶ Procesando PNFP
▶ Procesando PNRG
▶ Procesando PODD
▶ Procesando POOL
▶ Procesando POWL
▶ Procesando PPG
▶ Procesando PPIH
▶ Procesando P